# QFF 2026 PS-Q1 — Four-Qubit VQC

This notebook follows the earlier PS-Q1 brief's 200-sample synthetic dataset. The later consolidated guide describes an ECG/EEG version, so confirm the governing specification with the organizers before final submission.

Training is ideal-statevector training. The A/B comparison compiles the same trained logical circuit for each processor and measures circuit resources; it does not claim noisy A/B classification accuracy.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from src.problem import make_dataset
from src.vqc import train_vqc, predict, bind_circuit_for_sample
from src.processors import load_processor_config
from src.routing import transpile_for_processor
from src.metrics import circuit_metrics, save_metrics_csv
from src.visualization import plot_coupling_graph, plot_training_loss, plot_depth_comparison

ROOT = Path.cwd()
(ROOT / "results/tables").mkdir(parents=True, exist_ok=True)
(ROOT / "results/figures").mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT.resolve())

## 1. Generate the specified dataset

In [ ]:
X_train, X_test, y_train, y_test, scaler = make_dataset(random_state=42)
print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)
print("Train class counts:", np.bincount(y_train), "| Test class counts:", np.bincount(y_test))

## 2. Train and evaluate the VQC

In [ ]:
model = train_vqc(X_train, y_train, maxiter=35, seed=42)
y_pred = predict(model, X_test)
classification = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1": float(f1_score(y_test, y_pred, zero_division=0)),
    "final_training_loss": float(model["final_loss"]),
    "optimizer_success": model["optimizer_success"],
    "optimizer_message": model["optimizer_message"],
}
classification

In [ ]:
plot_training_loss(model["loss_history"], ROOT / "results/figures/training_loss.png")
print("Saved training loss plot.")

## 2B. Classical baseline (compatibility check)

In [ ]:
# A classical baseline is useful for judging whether the VQC adds value.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

baseline = LogisticRegression(random_state=42, max_iter=1000)
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
classical_metrics = {
    "accuracy": float(accuracy_score(y_test, baseline_pred)),
    "precision": float(precision_score(y_test, baseline_pred, zero_division=0)),
    "recall": float(recall_score(y_test, baseline_pred, zero_division=0)),
    "f1": float(f1_score(y_test, baseline_pred, zero_division=0)),
}
print("VQC metrics:", classification)
print("Logistic-regression baseline:", classical_metrics)

## 3. Load Processor A and B and visualize their coupling graphs

In [ ]:
processor_a = load_processor_config(ROOT / "processors/processor_A.json")
processor_b = load_processor_config(ROOT / "processors/processor_B.json")
for config in [processor_a, processor_b]:
    print(config["name"], "| qubits:", config["num_qubits"], "| edges:", config["coupling_map"])
    safe_name = config["name"].replace(" ", "_")
    plot_coupling_graph(config["graph"], config["name"] + " coupling map",
        ROOT / f"results/figures/{safe_name}_coupling.png")

## 4. Compile the same trained logical circuit on both processors

In [ ]:
# Bind the same held-out sample and trained weights before compiling.
logical = bind_circuit_for_sample(model, X_test[0])
compiled_a = transpile_for_processor(logical, processor_a)
compiled_b = transpile_for_processor(logical, processor_b)
rows = [
    circuit_metrics(compiled_a, processor_a["name"], processor_a["num_qubits"]),
    circuit_metrics(compiled_b, processor_b["name"], processor_b["num_qubits"]),
]
comparison = pd.DataFrame(rows)
comparison

In [ ]:
save_metrics_csv(rows, ROOT / "results/tables/processor_comparison.csv")
plot_depth_comparison(rows, ROOT / "results/figures/circuit_depth_comparison.png")
summary = {
    "problem_statement": "PS-Q1 VQC synthetic-data variant",
    "dataset": {"samples": 200, "features": 4, "informative": 3, "random_state": 42, "train_samples": 160, "test_samples": 40},
    "classification_metrics": classification,
    "classical_baseline_metrics": classical_metrics,
    "processor_comparison": rows,
    "transpiler": {"optimization_level": 1, "seed": 42},
    "note": "Ideal statevector training; compiled resource comparison only; no noisy A/B accuracy claimed."
}
with open(ROOT / "results/tables/summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print("Saved result tables and figures.")

## 5. Analysis questions

After reviewing the generated results, explain:
- Which processor has lower compiled circuit depth?
- How do CX counts and SWAP counts differ?
- What test accuracy, precision, recall and F1 did the model achieve?
- What are the limitations of ideal statevector training?
- Does the authoritative PS-Q1 specification require noisy sampled A/B evaluation?

Do not claim that lower circuit depth automatically means higher classification accuracy. Report measured results only.